# Customer Churn Prediction — Telecom

This project predicts which telecom customers are likely to cancel their subscription (churn), using the Telco Customer Churn dataset (7,043 customers, 21 features).

Instead of just building a classifier, I wanted to answer a real business question: which customers are about to leave, why, and how much revenue is at risk if we don't act.

Libraries used: Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, XGBoost, SHAP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import sklearn
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.rcParams["figure.figsize"] = [10,5]
# Ignore warnings

import warnings
# Set the warning filter to ignore FutureWarning
warnings.simplefilter(action = "ignore", category = FutureWarning)

### Loading the data and taking a first look

In [ ]:
full_data = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
# Data shape
print('train data:',full_data.shape)

In [ ]:
# View first few rows
full_data.head(5)

In [ ]:
full_data.info()

### Checking for missing/messy data

`TotalCharges` looked numeric but was actually stored as text, with some blank entries hiding inside it (new customers with 0 tenure). Checking for that before doing anything else.

In [ ]:
full_data.isin([" ", "", "NA", "NaN"]).sum() + full_data.isnull().sum()

In [ ]:
pd.to_numeric(full_data['TotalCharges'].str.strip(), errors='coerce').describe()

Converting `TotalCharges` to a proper number and filling the blanks with 0 (these are customers who just joined, so 0 total spend makes sense).

In [ ]:
full_data["TotalCharges"] = pd.to_numeric(
    full_data["TotalCharges"].str.strip(), errors="coerce"
).fillna(0)

In [ ]:
full_data.head(5)

## Exploratory Data Analysis

Looking for patterns in who churns and who doesn't, before building any model.

In [ ]:
full_data[["tenure", "MonthlyCharges", "TotalCharges"]].plot(
    kind="box", subplots=True, layout=(1, 3), figsize=(12, 4)
)

Overall churn rate — about a quarter of customers have left.

In [ ]:
full_data['Churn'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])

Churn by contract type — this turned out to be the biggest signal in the whole dataset.

In [ ]:
sns.countplot(data=full_data, x='Contract', hue='Churn', palette=['#2ecc71', '#e74c3c'])

Churn by internet service type.

In [ ]:
sns.countplot(
    data=full_data,
    x="InternetService",
    hue="Churn",
    palette=["#2ecc71", "#e74c3c"],
)

Zooming into fiber customers specifically — checking if having tech support changes their churn rate.

In [ ]:
# Fiber Optic users ka Tech Support & Churn comparison
sns.catplot(
    data=full_data[full_data["InternetService"] == "Fiber optic"],
    x="OnlineSecurity",
    hue="Churn",
    kind="count",
    palette=["#2ecc71", "#e74c3c"],
)

In [ ]:
# Fiber Optic users ka Tech Support & Churn comparison
sns.catplot(
    data=full_data[full_data["InternetService"] == "Fiber optic"],
    x="TechSupport",
    hue="Churn",
    kind="count",
    palette=["#2ecc71", "#e74c3c"],
)

Churn by payment method.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=full_data, y='PaymentMethod', hue='Churn', palette=['#2ecc71', '#e74c3c'], ax=axes[0])
sns.histplot(data=full_data, x='tenure', hue='Churn', kde=True, palette=['#2ecc71', '#e74c3c'], ax=axes[1])

## Feature Engineering

Dropping `customerID` (just an identifier, no predictive value) and converting `Churn` into 1/0 so models can use it.

In [ ]:
# 1. customerID drop karein aur Churn ko 1/0 banayein
full_data_clean = full_data.drop(columns=['customerID']).copy()
full_data_clean['Churn'] = full_data_clean['Churn'].map({'Yes': 1, 'No': 0})



Adding one engineered feature — average monthly spend — since raw `TotalCharges` mixes together how long someone's been a customer and how much they pay.

In [ ]:
# Feature Engineering — naya signal add karna
full_data_clean['avg_monthly_spend'] = full_data_clean['TotalCharges'] / (full_data_clean['tenure'] + 1)

full_data_clean['tenure_group'] = pd.cut(
    full_data_clean['tenure'],
    bins=[-1, 12, 24, 48, 72],
    labels=['0-12mo', '13-24mo', '25-48mo', '49-72mo']
)

service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']
full_data_clean['service_count'] = (full_data_clean[service_cols] == 'Yes').sum(axis=1)

print(full_data_clean[['avg_monthly_spend', 'tenure_group', 'service_count']].head())

One-hot encoding the categorical columns so they're usable by the models.

In [ ]:
# 2. Categorical columns ko One-Hot Encode karein
X_encoded = pd.get_dummies(full_data_clean.drop(columns=['Churn']), drop_first=True)
y = full_data_clean['Churn']

print("Original Shape:", full_data.shape)
print("Encoded Features Shape:", X_encoded.shape)

In [ ]:
print("Old Variable (full_data) Columns:", len(full_data.columns))
print("New Variable (full_data_clean) Columns:", len(full_data_clean.columns))

### Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print("Training Rows:", X_train.shape[0])
print("Testing Rows:", X_test.shape[0])

In [ ]:
import matplotlib.pyplot as plt

parts = ['X_train', 'y_train', 'X_test', 'y_test']
counts = [len(X_train), len(y_train), len(X_test), len(y_test)]

bars = plt.bar(parts, counts, color=['blue', 'green', 'orange', 'red'])
plt.bar_label(bars)  # Labels/numbers display karne ke liye
plt.show()

## Model 1: Logistic Regression (baseline)

Starting simple to have something to compare the fancier models against.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Binary Classification Model Train karein
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

# 2. Test data par Prediction aur Accuracy
y_pred_log = log_model.predict(X_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_log, cmap="Blues")

## Model 2: XGBoost

Trying a stronger model to see how much it improves over the baseline.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# 1. XGBoost Model train karein
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# 2. Predictions aur Classification Report
y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

### Fixing class imbalance

Only about 26% of customers churn, so a plain model tends to get lazy and just predict "stays" most of the time. Using `scale_pos_weight` to make XGBoost pay more attention to the churn class — missing a real churner is more costly to the business than a false alarm.

In [ ]:
# Scale Pos Weight ke saath XGBoost (Class Imbalance Fix)
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_balanced = XGBClassifier(scale_pos_weight=ratio, random_state=42, eval_metric='logloss')
xgb_balanced.fit(X_train, y_train)

y_pred_xgb_bal = xgb_balanced.predict(X_test)
print(classification_report(y_test, y_pred_xgb_bal))

## Model 3: Random Forest (balanced)

Comparing one more model type before deciding which one to actually ship.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import ConfusionMatrixDisplay

# 1. Balanced XGBoost Model
final_model = XGBClassifier(scale_pos_weight=2.8, random_state=42)
final_model.fit(X_train, y_train)

# 2. Confusion Matrix Check
ConfusionMatrixDisplay.from_predictions(y_test, final_model.predict(X_test), cmap="Blues")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay

# 1. Balanced Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train, y_train)

# 2. Confusion Matrix Check
ConfusionMatrixDisplay.from_predictions(y_test, rf_model.predict(X_test), cmap="Blues")

Adjusting the decision threshold — by default models use 0.5, but shifting this lets me trade off precision vs recall depending on what matters more for this business case (catching churners).

In [ ]:
# Threshold Adjust (0.6 Threshold)
y_probs = rf_model.predict_proba(X_test)[:, 1]
y_pred_adjusted = (y_probs >= 0.5).astype(int)

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_adjusted, cmap="Blues")

### Hyperparameter tuning

Using RandomizedSearchCV to find better Random Forest settings instead of guessing.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 8, 10],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced']
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    n_iter=10,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=-1,      # <- YEH ADD KIYA: saare CPU cores use karega, bohat fast hoga
    verbose=2        # <- YEH ADD KIYA: progress dikhega, "stuck" nahi lagega
)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best F1:", search.best_score_)

## Model Explainability (SHAP)

A prediction alone ("this customer has 80% churn risk") isn't very useful to a business team on its own — they need to know *why*. SHAP breaks down each prediction into which features pushed the risk up or down.

In [ ]:
!pip install shap -q

In [ ]:
import shap

# TreeExplainer specifically tree-based models (RF, XGBoost) ke liye fast hai
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# Binary classification mein shap_values[1] = "churn" class ke explanations
shap.summary_plot(shap_values[:, :, 1], X_test, plot_type="bar")

Looking at a single high-risk customer and seeing exactly which factors are driving their score up.

In [ ]:
# Ek specific customer choose karo (jaise test set ka pehla high-risk customer)
customer_idx = 0

shap.force_plot(
    explainer.expected_value[1],
    shap_values[customer_idx, :, 1],
    X_test.iloc[customer_idx],
    matplotlib=True
)

Turning that into a reusable function — top 3 reasons behind any customer's churn risk, in plain terms.

In [ ]:
def get_top_reasons(idx, n=3):
    row_shap = shap_values[idx, :, 1]
    feature_names = X_test.columns
    top_idx = np.argsort(np.abs(row_shap))[::-1][:n]
    reasons = [(feature_names[i], round(row_shap[i], 3)) for i in top_idx]
    return reasons

print(get_top_reasons(0))

## Scoring the Full Test Set & Saving the Model

Generating churn probabilities for every customer, then saving the trained model so the Streamlit app can load it without retraining.

In [ ]:
# Poore dataset (test set) ke liye churn probability nikalo
all_probs = rf_model.predict_proba(X_test)[:, 1]

results_df = X_test.copy()
results_df['Churn_Probability'] = all_probs
results_df['MonthlyCharges'] = full_data.loc[X_test.index, 'MonthlyCharges'].values

# Customer Value proxy — TotalCharges ka istemal karte hain (ab tak ka total kharcha)
results_df['CustomerValue'] = full_data.loc[X_test.index, 'TotalCharges'].values

# Revenue-at-Risk
results_df['RevenueAtRisk'] = results_df['MonthlyCharges'] * results_df['Churn_Probability']

# Risk Level
results_df['RiskLevel'] = pd.cut(
    results_df['Churn_Probability'],
    bins=[0, 0.35, 0.65, 1.0],
    labels=['Low', 'Medium', 'High']
)

# Retention Priority — Risk x Value dono combine
def assign_priority(row):
    high_value = row['CustomerValue'] > results_df['CustomerValue'].median()
    if row['RiskLevel'] == 'High' and high_value:
        return 'Priority 1 - Urgent'
    elif row['RiskLevel'] == 'High' and not high_value:
        return 'Priority 3 - Low-cost automation'
    elif row['RiskLevel'] == 'Medium' and high_value:
        return 'Priority 2 - Loyalty outreach'
    else:
        return 'Priority 4 - Monitor'

results_df['RetentionPriority'] = results_df.apply(assign_priority, axis=1)

# Sabse zyada revenue-at-risk wale top 20 customers dikhao
results_df.sort_values('RevenueAtRisk', ascending=False).head(20)

In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)

joblib.dump(rf_model, 'models/churn_model.pkl')
joblib.dump(X_encoded.columns.tolist(), 'models/model_columns.pkl')

print("Saved! Ab yeh files Colab ke 'models' folder mein hain.")

## Conclusion

The final model (balanced Random Forest / XGBoost) reaches **76.1% accuracy**, but the more important number here is **recall: 78.3%** — meaning it correctly flags about 4 out of every 5 customers who are actually going to churn. ROC-AUC came out to **84.4%**, which is solid for this kind of business dataset.

**Key drivers of churn:**
- Month-to-month contracts churn at **42.7%**, versus only **2.8%** for two-year contracts — by far the strongest signal in the data
- Fiber customers without a tech-support add-on churn well above average
- Newer customers (low tenure) are far more likely to leave than long-term ones

**Why I optimized for recall over precision:** in churn prevention, missing an actual churner costs the business real, permanent revenue. Flagging a loyal customer by mistake just costs one unnecessary retention offer. That asymmetry is why the model is deliberately tuned to catch more churners, even at the cost of some false alarms.

**What I'd try next:** more feature engineering around service bundles, testing SMOTE instead of class weighting, and calibrating the probability outputs so the risk scores are more directly interpretable as real probabilities.